<a href="https://colab.research.google.com/github/Sanjeev1654/AI-Driven-Stock-Sentiment-Engine-/blob/master/Portfolio_Optimization_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from datetime import datetime
import numpy as np
import pandas as pd
import requests
from io import StringIO
import scipy.optimize as optimization
import matplotlib.pyplot as plt

NUM_TRADING_DAYS = 252
DEFAULT_RISK_FREE = 0.015
RANDOM_SEED = 42

In [ ]:
STOOQ_ALIASES = {
    '^GSPC': '^spx',
}


def _to_stooq_symbol(ticker):
    if ticker in STOOQ_ALIASES:
        return STOOQ_ALIASES[ticker]
    if ticker.startswith('^'):
        return ticker.lower()
    return f"{ticker.lower()}.us"


def _stooq_download_one(ticker, start, end):
    symbol = _to_stooq_symbol(ticker)
    url = (f"https://stooq.com/q/d/l/?s={symbol}&d1={start.replace('-', '')}"
           f"&d2={end.replace('-', '')}&i=d")
    headers = {
        # Stooq blocks requests that look bot-like (default python-requests
        # UA gets a 404, not a 403), so pretend to be a normal browser.
        'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                        'AppleWebKit/537.36 (KHTML, like Gecko) '
                        'Chrome/124.0.0.0 Safari/537.36'),
        'Accept': 'text/csv,*/*',
    }
    resp = requests.get(url, headers=headers, timeout=20)
    if resp.status_code == 404:
        # Fall back to full history (no date range) -- some symbol/date-range
        # combinations 404 even when the symbol itself is valid.
        fallback_url = f"https://stooq.com/q/d/l/?s={symbol}&i=d"
        resp = requests.get(fallback_url, headers=headers, timeout=20)
    resp.raise_for_status()
    text = resp.text.strip()

    if not text:
        raise RuntimeError(f"Stooq returned an empty response for '{ticker}' (symbol '{symbol}'). URL: {url}")

    df = pd.read_csv(StringIO(text))
    # Normalize column names case-insensitively -- Stooq's schema is stable
    # (Date,Open,High,Low,Close,Volume) but be defensive about casing/whitespace.
    df.columns = [c.strip() for c in df.columns]
    col_map = {c.lower(): c for c in df.columns}
    if 'date' not in col_map or 'close' not in col_map:
        # This is the real failure mode: Stooq didn't give us a price CSV at
        # all -- usually a rate-limit notice, an HTML error page, or a
        # "no data" message. Show exactly what came back so it's fixable.
        preview = text[:300].replace('\n', ' | ')
        raise RuntimeError(
            f"Stooq did not return price data for '{ticker}' (symbol '{symbol}').\n"
            f"HTTP status: {resp.status_code}\n"
            f"URL tried: {url}\n"
            f"Response preview: {preview}"
        )

    date_col = col_map['date']
    close_col = col_map['close']
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.set_index(date_col).sort_index()
    df = df.loc[(df.index >= pd.Timestamp(start)) & (df.index <= pd.Timestamp(end))]
    return df[close_col].rename(ticker)


def download_prices(tickers, start, end=None, auto_adjust=True, progress=False, source='stooq'):
    """Download adjusted close prices.

    source='stooq' (default): free, no API key, no extra package beyond
        `requests` (already in the standard scientific-Python stack).
    source='yfinance': original path, used only if you have yfinance working.
    """
    end = end or datetime.today().strftime('%Y-%m-%d')
    if isinstance(tickers, str):
        tickers = [tickers]

    if source == 'yfinance':
        import yfinance as yf
        raw = yf.download(tickers, start=start, end=end, auto_adjust=auto_adjust, progress=progress)
        if raw is None or raw.empty:
            raise RuntimeError("yfinance returned no data. Check tickers/date range/Internet.")
        if isinstance(raw.columns, pd.MultiIndex):
            for primary in ('Close', 'Adj Close', 'AdjClose', 'close', 'adjclose'):
                if primary in raw.columns.get_level_values(0):
                    prices = raw[primary]
                    break
            else:
                prices = raw
        else:
            prices = raw
        if isinstance(prices, pd.Series):
            prices = prices.to_frame(name=tickers[0])
        return prices.dropna(axis=1, how='all')

    # --- Stooq path ---
    series_list = []
    for t in tickers:
        s = _stooq_download_one(t, start, end)
        series_list.append(s)
    prices = pd.concat(series_list, axis=1)
    return prices.dropna(axis=1, how='all')


def prices_logreturns(prices_df):
    return np.log(prices_df / prices_df.shift(1)).dropna()

In [ ]:
def simple_to_annual_metrics(daily_simple):
    if daily_simple.empty:
        return np.nan, np.nan
    cumulative = (1 + daily_simple).prod() - 1
    days = daily_simple.shape[0]
    cagr = (1 + cumulative) ** (NUM_TRADING_DAYS / days) - 1
    ann_vol = daily_simple.std() * np.sqrt(NUM_TRADING_DAYS)
    return cagr, ann_vol


def sharpe_ratio(daily_simple, rf=DEFAULT_RISK_FREE):
    cagr, vol = simple_to_annual_metrics(daily_simple)
    if vol == 0 or np.isnan(vol):
        return np.nan
    return (cagr - rf) / vol


def sortino_ratio(daily_simple, rf=DEFAULT_RISK_FREE):
    if daily_simple.empty:
        return np.nan
    downside = daily_simple[daily_simple < 0]
    if downside.empty:
        return np.nan
    cagr, _ = simple_to_annual_metrics(daily_simple)
    dd = downside.std() * np.sqrt(NUM_TRADING_DAYS)
    if dd == 0:
        return np.nan
    return (cagr - rf) / dd


def max_drawdown(daily_simple):
    if daily_simple.empty:
        return np.nan
    cum = (1 + daily_simple).cumprod()
    peak = cum.cummax()
    drawdown = (cum - peak) / peak
    return drawdown.min()


def historical_var(daily_simple, alpha=0.95):
    if daily_simple.empty:
        return np.nan
    return -np.percentile(daily_simple, 100 * (1 - alpha))


def historical_cvar(daily_simple, alpha=0.95):
    if daily_simple.empty:
        return np.nan
    cutoff = np.percentile(daily_simple, 100 * (1 - alpha))
    tail = daily_simple[daily_simple <= cutoff]
    if tail.empty:
        return 0.0
    return -tail.mean()


def summarize(daily_simple, name, rf=DEFAULT_RISK_FREE):
    cagr, vol = simple_to_annual_metrics(daily_simple)
    return {
        'Name': name,
        'CAGR': cagr,
        'Annual Vol': vol,
        'Sharpe': sharpe_ratio(daily_simple, rf),
        'Sortino': sortino_ratio(daily_simple, rf),
        'Max Drawdown': max_drawdown(daily_simple),
        'VaR(95%)': historical_var(daily_simple, 0.95),
        'CVaR(95%)': historical_cvar(daily_simple, 0.95),
    }

In [ ]:
def simulate_random_portfolios(returns_log_df, num_portfolios=8000, rf=DEFAULT_RISK_FREE, seed=RANDOM_SEED):
    """Monte Carlo sampling of the weight simplex. Used for:
       (a) a fallback if SLSQP fails, and
       (b) reporting the optimizer's percentile rank + plotting the frontier,
       so this is no longer a decorative step in the pipeline."""
    rng = np.random.default_rng(seed)
    n = returns_log_df.shape[1]
    mean_daily = returns_log_df.mean()
    cov_annual = returns_log_df.cov() * NUM_TRADING_DAYS

    rows = []
    for _ in range(int(num_portfolios)):
        w = rng.random(n)
        w /= w.sum()
        ann_return = float(np.sum(mean_daily * w) * NUM_TRADING_DAYS)
        ann_vol = float(np.sqrt(np.dot(w.T, cov_annual.dot(w))))
        sharpe = (ann_return - rf) / ann_vol if ann_vol != 0 else 0.0
        rows.append((ann_return, ann_vol, sharpe, w))
    return pd.DataFrame(rows, columns=['Return', 'Volatility', 'Sharpe', 'Weights'])


def optimize_max_sharpe(returns_log_df, rf=DEFAULT_RISK_FREE):
    def neg_sharpe(weights, returns_df):
        mean_daily = returns_df.mean()
        cov_annual = returns_df.cov() * NUM_TRADING_DAYS
        ann_return = np.sum(mean_daily * weights) * NUM_TRADING_DAYS
        ann_vol = np.sqrt(np.dot(weights.T, cov_annual.dot(weights)))
        if ann_vol == 0:
            return 1e6
        return -((ann_return - rf) / ann_vol)

    n = returns_log_df.shape[1]
    bounds = tuple((0.0, 1.0) for _ in range(n))
    cons = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1},)
    init = np.ones(n) / n
    res = optimization.minimize(neg_sharpe, init, args=(returns_log_df,), method='SLSQP',
                                 bounds=bounds, constraints=cons,
                                 options={'maxiter': 1000, 'ftol': 1e-9})
    return res


def fit_weights(logrets_window, portfolios_df=None, rf=DEFAULT_RISK_FREE):
    """Fit optimal weights using ONLY the data in logrets_window (a slice
    that ends at the rebalance date, never later)."""
    opt_res = optimize_max_sharpe(logrets_window, rf)
    if opt_res.success:
        w = np.array(opt_res.x, dtype=float)
    elif portfolios_df is not None:
        best_idx = portfolios_df['Sharpe'].idxmax()
        w = np.array(portfolios_df.loc[best_idx, 'Weights'], dtype=float)
    else:
        w = np.ones(logrets_window.shape[1]) / logrets_window.shape[1]
    w = w / w.sum()
    return w

In [ ]:
def walk_forward_backtest(prices_df, lookback_years=3, rebalance_frequency='M',
                           transaction_cost=0.0005, rf=DEFAULT_RISK_FREE,
                           num_portfolios=8000, seed=RANDOM_SEED, verbose=False):
    """
    Genuinely out-of-sample backtest.

    At each rebalance date t, weights are optimized using ONLY price history
    from [t - lookback_years, t). The portfolio then trades forward, unchanged,
    until the next rebalance date -- at which point it is re-optimized using
    the (now-extended) trailing window, again with zero knowledge of the
    future. No single set of weights is ever evaluated on the same data it
    was fit on.
    """
    logrets_full = prices_logreturns(prices_df)
    simple_full = np.expm1(logrets_full)
    dates = simple_full.index

    if rebalance_frequency == 'M':
        rebal_dates = sorted(pd.Series(dates).groupby(pd.Series(dates).dt.to_period('M')).first().values)
    elif rebalance_frequency == 'Q':
        rebal_dates = sorted(pd.Series(dates).groupby(pd.Series(dates).dt.to_period('Q')).first().values)
    else:
        rebal_dates = sorted(pd.Series(dates).groupby(pd.Series(dates).dt.to_period('M')).first().values)
    rebal_dates = pd.to_datetime(rebal_dates)

    lookback = pd.DateOffset(years=lookback_years)

    current_weights = None
    prev_weights = None
    capital = 1.0
    daily_port, date_list, weight_history = [], [], []
    cumulative_turnover = 0.0
    frontier_rank = None  # percentile rank of the optimizer vs Monte Carlo, captured once

    for dt in dates:
        if (current_weights is None) and (dt < rebal_dates[0] + lookback):
            # Not enough trailing history yet to fit a first out-of-sample window; skip.
            continue

        is_rebal = dt in set(rebal_dates)
        if is_rebal or current_weights is None:
            window_start = dt - lookback
            train_window = logrets_full.loc[(logrets_full.index >= window_start) & (logrets_full.index < dt)]
            if len(train_window) < 60:  # need a minimum sample to fit covariance sensibly
                if current_weights is None:
                    continue
            else:
                portfolios_df = simulate_random_portfolios(train_window, num_portfolios, rf, seed)
                target = fit_weights(train_window, portfolios_df, rf)

                if frontier_rank is None:
                    opt_sharpe = None
                    opt_res = optimize_max_sharpe(train_window, rf)
                    if opt_res.success:
                        opt_sharpe = -opt_res.fun
                    if opt_sharpe is not None:
                        frontier_rank = float((portfolios_df['Sharpe'] < opt_sharpe).mean() * 100)

                if current_weights is None:
                    current_weights = target.copy()
                    prev_weights = target.copy()
                else:
                    turnover = np.sum(np.abs(target - prev_weights)) / 2.0
                    cost = capital * transaction_cost * turnover
                    capital -= cost
                    cumulative_turnover += turnover
                    current_weights = target.copy()
                    prev_weights = target.copy()
                weight_history.append((dt, current_weights.copy()))

        todays_rets = simple_full.loc[dt].values
        port_ret = float(np.dot(current_weights, todays_rets))
        capital *= (1 + port_ret)
        daily_port.append(port_ret)
        date_list.append(dt)

        denom = np.sum(current_weights * (1 + todays_rets))
        current_weights = current_weights if denom == 0 else (current_weights * (1 + todays_rets)) / denom
        prev_weights = current_weights.copy()

    port_series = pd.Series(daily_port, index=date_list).sort_index()
    details = {
        'final_capital': capital,
        'cumulative_turnover': cumulative_turnover,
        'transaction_cost_rate': transaction_cost,
        'rebalance_frequency': rebalance_frequency,
        'lookback_years': lookback_years,
        'oos_start': date_list[0] if date_list else None,
        'oos_end': date_list[-1] if date_list else None,
        'optimizer_sharpe_percentile_vs_montecarlo': frontier_rank,
        'weight_history': weight_history,
    }
    return port_series, details

In [ ]:
def run_pipeline(tickers, start_date='2012-01-01', end_date=None,
                  lookback_years=3, rebalance_frequency='M',
                  transaction_cost=0.0005, risk_free=DEFAULT_RISK_FREE,
                  num_portfolios=8000, csv_out='optimal_weights_history.csv',
                  data_source='yfinance'):

    prices = download_prices(tickers, start=start_date, end=end_date, source=data_source)

    port_daily, details = walk_forward_backtest(
        prices, lookback_years=lookback_years, rebalance_frequency=rebalance_frequency,
        transaction_cost=transaction_cost, rf=risk_free, num_portfolios=num_portfolios)

    bench = download_prices(['^GSPC'], start=start_date, end=end_date, source=data_source)
    bench_series = bench.iloc[:, 0] if isinstance(bench, pd.DataFrame) else bench
    bench_log = np.log(bench_series / bench_series.shift(1)).dropna()
    bench_simple = np.expm1(bench_log).reindex(port_daily.index).dropna()
    port_daily = port_daily.reindex(bench_simple.index).dropna()

    port_summary = summarize(port_daily, 'Optimal Portfolio (out-of-sample)', risk_free)
    bench_summary = summarize(bench_simple, 'S&P 500 (Benchmark)', risk_free)

    weight_rows = []
    for dt, w in details['weight_history']:
        row = {'date': dt}
        row.update({t: wt for t, wt in zip(prices.columns, w)})
        weight_rows.append(row)
    weights_over_time = pd.DataFrame(weight_rows)
    weights_over_time.to_csv(csv_out, index=False)

    return {
        'weights_over_time': weights_over_time,
        'port_summary': port_summary,
        'bench_summary': bench_summary,
        'csv': os.path.abspath(csv_out),
        'backtest_details': {k: v for k, v in details.items() if k != 'weight_history'},
    }

In [ ]:
if __name__ == '__main__':
    TICKERS = ['MSFT', 'KO', 'TSLA', 'NFLX', 'JNJ']
    RESULT = run_pipeline(
        TICKERS,
        start_date='2012-01-01',   # extra years upfront feed the first lookback window
        end_date=None,
        lookback_years=3,          # fit on trailing 3 years only
        rebalance_frequency='M',
        transaction_cost=0.0005,
        risk_free=0.015,
        num_portfolios=8000,
        csv_out='optimal_weights_history.csv',
        data_source='yfinance',
    )

    print("\nOut-of-sample portfolio summary:")
    for k, v in RESULT['port_summary'].items():
        if k == 'Name':
            continue
        print(f"  {k}: {v}")

    print("\nBenchmark (S&P 500) summary:")
    for k, v in RESULT['bench_summary'].items():
        if k == 'Name':
            continue
        print(f"  {k}: {v}")

    print("\nBacktest details:")
    for k, v in RESULT['backtest_details'].items():
        print(f"  {k}: {v}")

    print(f"\nRolling weights history saved to: {RESULT['csv']}")


Out-of-sample portfolio summary:
  CAGR: 0.16810138142954334
  Annual Vol: 0.24855976203267444
  Sharpe: 0.6159540071068196
  Sortino: 0.7919214332326668
  Max Drawdown: -0.5047253482397938
  VaR(95%): 0.0243950973287197
  CVaR(95%): 0.03857020757955646

Benchmark (S&P 500) summary:
  CAGR: 0.12006100311559575
  Annual Vol: 0.17701481376091338
  Sharpe: 0.5935153159412824
  Sortino: 0.7274648064777514
  Max Drawdown: -0.3392496000265331
  VaR(95%): 0.01646322052135858
  CVaR(95%): 0.026836129244524977

Backtest details:
  final_capital: 6.022677739085506
  cumulative_turnover: 15.963558825080996
  transaction_cost_rate: 0.0005
  rebalance_frequency: M
  lookback_years: 3
  oos_start: 2015-01-05 00:00:00
  oos_end: 2026-08-21 00:00:00
  optimizer_sharpe_percentile_vs_montecarlo: 100.0

Rolling weights history saved to: /content/optimal_weights_history.csv
